<a href="https://colab.research.google.com/github/Pranayshukla0610/Exploratory-Data-Analysis-EDA-/blob/main/Real_Time_Finance_Analytic_Dashboard_using_Bokeh_Library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

from math import pi

from bokeh.io import curdoc
from bokeh.layouts import row, column
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    Div,
    Select
)

In [5]:
output_notebook()

In [6]:
BACKGROUND = '#0F172A'
CARD = '#1E293B'
TEXT = '#F8FAFC'

CHART_WIDTH = 650
CHART_HEIGHT = 400

In [7]:
ticker = 'NVDA'
data = yf.download(
    ticker,
    period='2y'
)

/tmp/ipykernel_6202/394394284.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1 of 1 completed


In [8]:
data.reset_index(inplace=True)

In [9]:
data.columns = data.columns.get_level_values(0)

In [10]:
data['Returns'] = data['Close'].pct_change()

In [11]:
data['MA20'] = data['Close'].rolling(20).mean()
data['MA50'] = data['Close'].rolling(50).mean()

In [12]:
data['Volatility'] = data['Returns'].rolling(20).std()

In [13]:
data['Upper'] = data['MA20'] + 2 * data['Volatility']
data['Lower'] = data['MA20'] - 2 * data['Volatility']

In [14]:
data.dropna(inplace=True)

In [15]:
source = ColumnDataSource(data)

In [16]:
latest_price = round(
    data['Close'].iloc[-1],
    2
)

In [17]:
daily_return = round(
    data['Returns'].iloc[-1] * 100,
    2
)

In [18]:
volatility = round(
    data['Volatility'].mean() * 100,
    2
)

In [19]:
investment = 10000

shares = investment / data['Close'].iloc[0]

In [20]:
portfolio_value = round(
    shares * data['Close'].iloc[-1],
    2
)

In [24]:
profit = round(
    portfolio_value - investment,
    2
)

In [21]:
# Sharpe Ratio
risk_free_rate = 0.02

sharpe_ratio = round(
    (
        data['Returns'].mean()*252
        - risk_free_rate
    )
    /
    (
        data['Returns'].std()
        * np.sqrt(252)
    ),
    2
)

In [22]:
def create_kpi(title, value, color):

    return Div(text=f"""
    <div style="
    background:{color};
    padding:20px;
    border-radius:16px;
    text-align:center;
    width:220px;
    height:120px;
    box-shadow:0px 4px 14px rgba(0,0,0,0.5);
    ">

    <h2 style="color:white;">
    {title}
    </h2>

    <h1 style="color:white;">
    {value}
    </h1>

    </div>
    """)

In [25]:
kpi1 = create_kpi(
    "Live Price",
    f"${latest_price}",
    "#2563EB"
)

kpi2 = create_kpi(
    "Daily Return",
    f"{daily_return}%",
    "#059669"
)

kpi3 = create_kpi(
    "Volatility",
    f"{volatility}%",
    "#DC2626"
)

kpi4 = create_kpi(
    "Portfolio Value",
    f"${portfolio_value}",
    "#7C3AED"
)

kpi5 = create_kpi(
    "Profit/Loss",
    f"${profit}",
    "#EA580C"
)

kpi6 = create_kpi(
    "Sharpe Ratio",
    sharpe_ratio,
    "#0891B2"
)

In [26]:
stock_selector = Select(
    title="Select Company",
    value="NVDA",
    options=[
        "NVDA",
        "MSFT",
        "TSLA",
        "AMZN",
        "META",
        "GOOGL"
    ]
)

In [27]:
inc = data['Close'] > data['Open']
dec = data['Open'] > data['Close']

w = 12*60*60*1000

p1 = figure(
    x_axis_type="datetime",
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    background_fill_color=CARD,
    border_fill_color=BACKGROUND,
    title="Candlestick Analysis"
)

p1.segment(
    data['Date'],
    data['High'],
    data['Date'],
    data['Low'],
    color="white"
)

p1.vbar(
    data.loc[inc,'Date'],
    w,
    data.loc[inc,'Open'],
    data.loc[inc,'Close'],
    fill_color="#10B981",
    line_color="#10B981"
)

p1.vbar(
    data.loc[dec,'Date'],
    w,
    data.loc[dec,'Open'],
    data.loc[dec,'Close'],
    fill_color="#EF4444",
    line_color="#EF4444"
)

GlyphRenderer(id='p1095', ...)

In [28]:
p2 = figure(
    x_axis_type="datetime",
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    background_fill_color=CARD,
    border_fill_color=BACKGROUND,
    title="Moving Average Analytics"
)

p2.line(
    data['Date'],
    data['Close'],
    line_width=2,
    color="white",
    legend_label="Close"
)

p2.line(
    data['Date'],
    data['MA20'],
    line_width=3,
    color="#3B82F6",
    legend_label="MA20"
)

p2.line(
    data['Date'],
    data['MA50'],
    line_width=3,
    color="#F59E0B",
    legend_label="MA50"
)

GlyphRenderer(id='p1176', ...)

In [29]:
p3 = figure(
    x_axis_type="datetime",
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    background_fill_color=CARD,
    border_fill_color=BACKGROUND,
    title="Volume Analysis"
)

p3.vbar(
    x=data['Date'],
    top=data['Volume'],
    width=w,
    color="#06B6D4"
)

GlyphRenderer(id='p1237', ...)

In [30]:
p4 = figure(
    x_axis_type="datetime",
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    background_fill_color=CARD,
    border_fill_color=BACKGROUND,
    title="Risk & Volatility"
)

p4.line(
    data['Date'],
    data['Volatility'],
    line_width=3,
    color="#EF4444"
)


GlyphRenderer(id='p1297', ...)

In [31]:
hist, edges = np.histogram(
    data['Returns'].dropna(),
    bins=40
)

p5 = figure(
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    background_fill_color=CARD,
    border_fill_color=BACKGROUND,
    title="Return Distribution"
)

p5.quad(
    top=hist,
    bottom=0,
    left=edges[:-1],
    right=edges[1:],
    fill_color="#8B5CF6",
    line_color="white"
)

GlyphRenderer(id='p1343', ...)

In [32]:
p6 = figure(
    x_axis_type="datetime",
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    background_fill_color=CARD,
    border_fill_color=BACKGROUND,
    title="Bollinger Bands"
)

p6.line(
    data['Date'],
    data['Close'],
    line_width=2,
    color="white",
    legend_label="Close"
)

p6.line(
    data['Date'],
    data['Upper'],
    line_width=2,
    color="#10B981",
    legend_label="Upper Band"
)

p6.line(
    data['Date'],
    data['Lower'],
    line_width=2,
    color="#EF4444",
    legend_label="Lower Band"
)

GlyphRenderer(id='p1424', ...)

In [33]:
plots = [p1,p2,p3,p4,p5,p6]

for p in plots:

    p.title.text_color = "white"

    p.xaxis.major_label_text_color = "white"
    p.yaxis.major_label_text_color = "white"

    p.xaxis.axis_label_text_color = "white"
    p.yaxis.axis_label_text_color = "white"

    p.grid.grid_line_color = "#334155"


In [34]:
title = Div(text="""
<div style="
background:linear-gradient(
90deg,
#111827,
#1E3A8A
);

padding:25px;
border-radius:18px;
box-shadow:0px 4px 14px rgba(0,0,0,0.5);
">

<h1 style="
color:white;
text-align:center;
font-size:40px;
font-family:Arial;
">
REAL-TIME FINANCIAL ANALYTICS DASHBOARD
</h1>

</div>
""")


In [35]:
def update(attr, old, new):

    global data

    ticker = stock_selector.value

    new_data = yf.download(
        ticker,
        period="2y"
    )

    new_data.reset_index(inplace=True)

    new_data.columns = new_data.columns.get_level_values(0)

    new_data['Returns'] = (
        new_data['Close'].pct_change()
    )

    new_data['MA20'] = (
        new_data['Close']
        .rolling(20)
        .mean()
    )

    new_data['MA50'] = (
        new_data['Close']
        .rolling(50)
        .mean()
    )

    new_data['Volatility'] = (
        new_data['Returns']
        .rolling(20)
        .std()
    )

    new_data.dropna(inplace=True)

    source.data = ColumnDataSource(
        new_data
    ).data

stock_selector.on_change(
    'value',
    update
)


In [36]:
dashboard = column(

    title,

    row(
        kpi1,
        kpi2,
        kpi3
    ),

    row(
        kpi4,
        kpi5,
        kpi6
    ),

    row(
        stock_selector
    ),

    row(
        p1,
        p2
    ),

    row(
        p3,
        p4
    ),

    row(
        p5,
        p6
    )
)

In [37]:
show(dashboard)

You are generating standalone HTML/JS output, but trying to use real Python
callbacks (i.e. with on_change or on_event). This combination cannot work.

Only JavaScript callbacks may be used with standalone output. For more
information on JavaScript callbacks with Bokeh, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/interaction/js_callbacks.html

Alternatively, to use real Python callbacks, a Bokeh server application may
be used. For more information on building and running Bokeh applications, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/server.html



In [38]:
from bokeh.io import output_file, save

output_file("dashboard.html")

save(dashboard)

show(dashboard)

You are generating standalone HTML/JS output, but trying to use real Python
callbacks (i.e. with on_change or on_event). This combination cannot work.

Only JavaScript callbacks may be used with standalone output. For more
information on JavaScript callbacks with Bokeh, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/interaction/js_callbacks.html

Alternatively, to use real Python callbacks, a Bokeh server application may
be used. For more information on building and running Bokeh applications, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/server.html



You are generating standalone HTML/JS output, but trying to use real Python
callbacks (i.e. with on_change or on_event). This combination cannot work.

Only JavaScript callbacks may be used with standalone output. For more
information on JavaScript callbacks with Bokeh, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/interaction/js_callbacks.html

Alternatively, to use real Python callbacks, a Bokeh server application may
be used. For more information on building and running Bokeh applications, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/server.html



You are generating standalone HTML/JS output, but trying to use real Python
callbacks (i.e. with on_change or on_event). This combination cannot work.

Only JavaScript callbacks may be used with standalone output. For more
information on JavaScript callbacks with Bokeh, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/interaction/js_callbacks.html

Alternatively, to use real Python callbacks, a Bokeh server application may
be used. For more information on building and running Bokeh applications, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/server.html



In [39]:
from google.colab import files

files.download("dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>